In [8]:
# -*- coding: utf-8 -*-
"""
Improved GRU4Rec Training with Debugging
"""

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm
import time
from pathlib import Path

# ========== CONFIGURATION ==========
class Config:
    # Paths
    data_path = '../data/processed/gru4rec_t/'
    checkpoint_path = './checkpoints/'
    
    # Model Architecture - SIMPLIFIED
    embedding_dim = 50  # Use embeddings instead of one-hot
    hidden_size = 50    # Reduced for small dataset
    num_layers = 1
    dropout_hidden = 0.2  # Reduced dropout
    dropout_embed = 0.0
    
    # Training
    batch_size = 128    # Smaller batch size
    learning_rate = 0.001  # Lower learning rate
    momentum = 0.0
    n_epochs = 30       # More epochs for small dataset
    
    # Loss function
    loss_type = 'cross-entropy'
    
    # Sampling
    n_sample = 512
    sample_alpha = 0.75
    
    # Evaluation
    eval_top_k = [5, 10, 20]
    
    # Device
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    
    # Seed
    seed = 42

config = Config()
Path(config.checkpoint_path).mkdir(parents=True, exist_ok=True)
torch.manual_seed(config.seed)
np.random.seed(config.seed)

print(f"Training on device: {config.device}")

# ========== DATA LOADING WITH DIAGNOSTICS ==========
class SessionDataset(Dataset):
    """Improved dataset with better handling"""
    
    def __init__(self, path, train=True, item2idx=None):
        print(f"\nLoading data from: {path}")
        self.df = pd.read_csv(path, sep='\t', dtype={'ItemId': np.int32})
        
        print(f"Raw data: {len(self.df)} rows, {self.df['SessionId'].nunique()} sessions")
        
        # Group by session
        self.sessions = []
        self.session_ids = []
        
        session_lengths = []
        for sid, group in self.df.groupby('SessionId'):
            items = group['ItemId'].values
            if len(items) >= 2:  # Minimum length
                self.sessions.append(items)
                self.session_ids.append(sid)
                session_lengths.append(len(items))
        
        print(f"Valid sessions (len>=2): {len(self.sessions)}")
        print(f"Session length stats - Mean: {np.mean(session_lengths):.1f}, "
              f"Median: {np.median(session_lengths):.0f}, "
              f"Max: {max(session_lengths)}")
        
        # Build or use provided item mapping
        if item2idx is None:
            all_items = self.df['ItemId'].unique()
            self.item2idx = {item: idx for idx, item in enumerate(sorted(all_items), start=1)}
            self.idx2item = {idx: item for item, idx in self.item2idx.items()}
            self.n_items = len(self.item2idx) + 1
            print(f"Created item mapping: {self.n_items-1} unique items")
        else:
            self.item2idx = item2idx
            self.idx2item = {idx: item for item, idx in self.item2idx.items()}
            self.n_items = len(self.item2idx) + 1
            
            # Check coverage
            dataset_items = set(self.df['ItemId'].unique())
            mapped_items = set(self.item2idx.keys())
            coverage = len(dataset_items & mapped_items) / len(dataset_items) * 100
            print(f"Item mapping coverage: {coverage:.1f}% ({len(dataset_items & mapped_items)}/{len(dataset_items)})")
    
    def __len__(self):
        return len(self.sessions)
    
    def __getitem__(self, idx):
        return self.sessions[idx], self.session_ids[idx]

print("="*60)
print("LOADING DATASETS")
print("="*60)

train_dataset = SessionDataset(config.data_path + 'movielens_train_tr.txt')
valid_dataset = SessionDataset(config.data_path + 'movielens_train_valid.txt', 
                               item2idx=train_dataset.item2idx)
test_dataset = SessionDataset(config.data_path + 'movielens_test.txt',
                              item2idx=train_dataset.item2idx)

config.n_items = train_dataset.n_items

# ========== IMPROVED COLLATE FUNCTION ==========
def make_collate_fn(dataset):
    def collate_fn(batch):
        """Enhanced collate with session context"""
        sessions, session_ids = zip(*batch)
        
        inputs, targets, masks = [], [], []
        
        for session in sessions:
            session_idx = [dataset.item2idx.get(item, 0) for item in session]
            
            # Create all input-target pairs
            for i in range(len(session_idx) - 1):
                inputs.append(session_idx[i])
                targets.append(session_idx[i + 1])
                masks.append(1 if session_idx[i] > 0 and session_idx[i+1] > 0 else 0)
        
        return (torch.LongTensor(inputs), 
                torch.LongTensor(targets),
                torch.FloatTensor(masks))
    return collate_fn

train_loader = DataLoader(train_dataset, batch_size=config.batch_size, 
                          shuffle=True, collate_fn=make_collate_fn(train_dataset))

# ========== IMPROVED MODEL ==========
class GRU4Rec(nn.Module):
    """Improved GRU4Rec with better initialization"""
    
    def __init__(self, n_items, hidden_size, num_layers=1, 
                 embedding_dim=50, dropout_hidden=0.2, dropout_embed=0.0):
        super(GRU4Rec, self).__init__()
        
        self.n_items = n_items
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.embedding_dim = embedding_dim
        
        # Embedding layer
        self.item_embedding = nn.Embedding(n_items, embedding_dim, padding_idx=0)
        nn.init.normal_(self.item_embedding.weight, 0, 0.01)
        self.embed_dropout = nn.Dropout(dropout_embed)
        
        # GRU
        self.gru = nn.GRU(embedding_dim, hidden_size, num_layers, 
                          dropout=dropout_hidden if num_layers > 1 else 0,
                          batch_first=True)
        
        self.dropout = nn.Dropout(dropout_hidden)
        
        # Output layer
        self.output = nn.Linear(hidden_size, n_items)
        nn.init.normal_(self.output.weight, 0, 0.01)
        nn.init.zeros_(self.output.bias)
    
    def forward(self, input_items, hidden=None):
        batch_size = input_items.size(0)
        
        if len(input_items.shape) == 1:
            input_items = input_items.unsqueeze(1)
        
        # Embedding
        embedded = self.item_embedding(input_items)
        embedded = self.embed_dropout(embedded)
        
        # GRU
        output, hidden = self.gru(embedded, hidden)
        output = self.dropout(output[:, -1, :])
        
        # Output
        logits = self.output(output)
        
        return logits, hidden

# ========== IMPROVED EVALUATION ==========
def evaluate_model(model, dataset, config, top_k_list=[5, 10, 20]):
    """Improved evaluation without train masking bug"""
    model.eval()
    
    hrs = {k: [] for k in top_k_list}
    mrrs = []
    ndcgs = {k: [] for k in top_k_list}
    
    with torch.no_grad():
        for session, session_id in tqdm(dataset, desc="Evaluating", disable=len(dataset)<100):
            if len(session) < 2:
                continue
            
            # Use all but last as input
            input_items = session[:-1]
            target_item = session[-1]
            
            # Convert to indices
            target_idx = dataset.item2idx.get(target_item, 0)
            if target_idx == 0:
                continue
            
            # Get prediction from last input
            last_input = dataset.item2idx.get(input_items[-1], 0)
            if last_input == 0:
                continue
            
            input_tensor = torch.LongTensor([last_input]).to(config.device)
            logits, _ = model(input_tensor)
            scores = logits[0].cpu().numpy()
            
            # Mask padding only
            scores[0] = -np.inf
            
            # Get top-K predictions
            top_k_max = max(top_k_list)
            top_indices = np.argsort(-scores)[:top_k_max]
            
            # Compute metrics
            for k in top_k_list:
                if target_idx in top_indices[:k]:
                    hrs[k].append(1.0)
                    rank = np.where(top_indices[:k] == target_idx)[0][0] + 1
                    ndcgs[k].append(1.0 / np.log2(rank + 1))
                else:
                    hrs[k].append(0.0)
                    ndcgs[k].append(0.0)
            
            # MRR
            if target_idx in top_indices:
                rank = np.where(top_indices == target_idx)[0][0] + 1
                mrrs.append(1.0 / rank)
            else:
                mrrs.append(0.0)
    
    results = {'MRR': np.mean(mrrs) if mrrs else 0.0}
    for k in top_k_list:
        results[f'HR@{k}'] = np.mean(hrs[k]) if hrs[k] else 0.0
        results[f'NDCG@{k}'] = np.mean(ndcgs[k]) if ndcgs[k] else 0.0
    
    return results

# ========== TRAINING ==========
print("\n" + "="*60)
print("INITIALIZING MODEL")
print("="*60)

model = GRU4Rec(
    n_items=config.n_items,
    hidden_size=config.hidden_size,
    num_layers=config.num_layers,
    embedding_dim=config.embedding_dim,
    dropout_hidden=config.dropout_hidden,
    dropout_embed=config.dropout_embed
).to(config.device)

optimizer = optim.Adam(model.parameters(), lr=config.learning_rate)
criterion = nn.CrossEntropyLoss()

print(f"Model Parameters: {sum(p.numel() for p in model.parameters()):,}")
print(f"Trainable Parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}")

print("\n" + "="*60)
print("STARTING TRAINING")
print("="*60)

best_valid_hr = 0.0
patience = 10
patience_counter = 0

for epoch in range(config.n_epochs):
    model.train()
    epoch_loss = 0.0
    n_batches = 0
    
    pbar = tqdm(train_loader, desc=f"Epoch {epoch+1}/{config.n_epochs}")
    for inputs, targets, masks in pbar:
        inputs = inputs.to(config.device)
        targets = targets.to(config.device)
        masks = masks.to(config.device)
        
        # Filter out masked items
        valid_mask = masks > 0
        if valid_mask.sum() == 0:
            continue
            
        inputs = inputs[valid_mask]
        targets = targets[valid_mask]
        
        # Forward
        logits, _ = model(inputs)
        loss = criterion(logits, targets)
        
        # Backward
        optimizer.zero_grad()
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        
        epoch_loss += loss.item()
        n_batches += 1
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    
    avg_loss = epoch_loss / max(n_batches, 1)
    
    # Validation every 5 epochs
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f"\n{'='*60}")
        print(f"Epoch {epoch+1} | Avg Loss: {avg_loss:.4f}")
        print(f"{'='*60}")
        
        valid_results = evaluate_model(model, valid_dataset, config, config.eval_top_k)
        
        print("Validation Results:")
        for metric, value in valid_results.items():
            print(f"  {metric}: {value:.4f}")
        
        # Save best model
        if valid_results['HR@10'] > best_valid_hr:
            best_valid_hr = valid_results['HR@10']
            patience_counter = 0
            torch.save({
                'epoch': epoch,
                'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'valid_results': valid_results,
            }, config.checkpoint_path + 'best_model.pt')
            print(f"✓ Best model saved! (HR@10: {best_valid_hr:.4f})")
        else:
            patience_counter += 1
        
        print()
        
        # Early stopping
        if patience_counter >= patience:
            print(f"Early stopping triggered after {epoch+1} epochs")
            break

# ========== TEST EVALUATION ==========
print("\n" + "="*60)
print("FINAL TEST EVALUATION")
print("="*60)

checkpoint = torch.load(config.checkpoint_path + 'best_model.pt')
model.load_state_dict(checkpoint['model_state_dict'])

test_results = evaluate_model(model, test_dataset, config, config.eval_top_k)

print("\nTest Results:")
print("-" * 40)
for metric, value in test_results.items():
    print(f"  {metric}: {value:.4f}")
print("-" * 40)

print("\n✓ Training complete!")

Training on device: cpu
LOADING DATASETS

Loading data from: ../data/processed/gru4rec_t/movielens_train_tr.txt
Raw data: 556806 rows, 5981 sessions
Valid sessions (len>=2): 5981
Session length stats - Mean: 93.1, Median: 57, Max: 1174
Created item mapping: 3125 unique items

Loading data from: ../data/processed/gru4rec_t/movielens_train_valid.txt
Raw data: 7031 rows, 23 sessions
Valid sessions (len>=2): 23
Session length stats - Mean: 305.7, Median: 259, Max: 792
Item mapping coverage: 100.0% (1885/1885)

Loading data from: ../data/processed/gru4rec_t/movielens_test.txt
Raw data: 10543 rows, 31 sessions
Valid sessions (len>=2): 31
Session length stats - Mean: 340.1, Median: 231, Max: 1417
Item mapping coverage: 100.0% (2245/2245)

INITIALIZING MODEL
Model Parameters: 331,026
Trainable Parameters: 331,026

STARTING TRAINING


Epoch 1/30: 100%|██████████| 47/47 [00:13<00:00,  3.41it/s, loss=7.6746]



Epoch 1 | Avg Loss: 7.9219
Validation Results:
  MRR: 0.0000
  HR@5: 0.0000
  NDCG@5: 0.0000
  HR@10: 0.0000
  NDCG@10: 0.0000
  HR@20: 0.0000
  NDCG@20: 0.0000



Epoch 2/30:  43%|████▎     | 20/47 [00:06<00:08,  3.24it/s, loss=7.3886]


KeyboardInterrupt: 